
# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 ตัวอย่างที่ 1: ใช้ NumPy คำนวณข้อมูล NDVI จากภาพ Raster


In [2]:

import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.7, 0.8, 0.6], [0.9, 0.5, 0.3], [0.4, 0.7, 0.2]])
red = np.array([[0.3, 0.4, 0.2], [0.5, 0.3, 0.1], [0.2, 0.3, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)


ค่า NDVI:
[[0.4        0.33333333 0.5       ]
 [0.28571429 0.25       0.5       ]
 [0.33333333 0.4        0.33333333]]



## 🔹 ตัวอย่างที่ 2: ใช้ NumPy คำนวณค่าเฉลี่ยและค่ามากสุดของ NDVI


In [3]:

print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")


ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25



## 🔹 ตัวอย่างที่ 3: ใช้ GeoPandas โหลดและวิเคราะห์ข้อมูล Shapefile


In [20]:

import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
# ข้อมูลจาก OpenDevelopment แม่น้ำโขง
gdf = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())


   Shape_Leng  Shape_Area        ADM1_EN        ADM1_TH ADM1_PCODE ADM1_REF  \
0    3.927244    0.275313  Amnat Charoen     อำนาจเจริญ       TH37     None   
1    1.739908    0.079210      Ang Thong        อ่างทอง       TH15     None   
2    2.417227    0.131339        Bangkok  กรุงเทพมหานคร       TH10     None   
3    4.414998    0.340784      Bueng Kan         บึงกาฬ       TH38     None   
4    8.701860    0.844537       Buri Ram      บุรีรัมย์       TH31     None   

  ADM1ALT1EN ADM1ALT2EN ADM1ALT1TH ADM1ALT2TH   ADM0_EN    ADM0_TH ADM0_PCODE  \
0       None       None       None       None  Thailand  ประเทศไทย         TH   
1       None       None       None       None  Thailand  ประเทศไทย         TH   
2       None       None       None       None  Thailand  ประเทศไทย         TH   
3       None       None       None       None  Thailand  ประเทศไทย         TH   
4       None       None       None       None  Thailand  ประเทศไทย         TH   

        date    validOn validTo  \
0 2


## 🔹 ตัวอย่างที่ 4: คำนวณพื้นที่ของแต่ละจังหวัด


In [22]:
# ตรวจสอบค่า CRS (Coordinate Reference System)
print(gdf.crs)

# หาก CRS เป็นแบบภูมิศาสตร์ (เช่น Lat/Lon) ให้แปลงเป็น CRS แบบฉายภาพ (Projected CRS) ก่อนคำนวณพื้นที่
# ตัวอย่าง: แปลงเป็น WGS 84 / UTM zone 47N (EPSG:32647) ซึ่งเหมาะสมกับประเทศไทย
# หรือใช้ EPSG:4326 (WGS 84) และคำนวณพื้นที่บนทรงรี

# ตรวจสอบว่ามีคอลัมน์ 'ADM1_TH' หรือไม่ หากไม่มี ให้ใช้ 'ADM1_EN'
name_column = 'ADM1_TH' if 'ADM1_TH' in gdf.columns else 'ADM1_EN'

if gdf.crs.is_geographic:
    print("CRS เป็นแบบภูมิศาสตร์ (Lat/Lon) จะทำการแปลงเป็น Projected CRS (WGS 84 / UTM zone 47N) เพื่อคำนวณพื้นที่ที่แม่นยำขึ้น")
    gdf_proj = gdf.to_crs(epsg=32647) # หรือเลือก EPSG code ที่เหมาะสมกับพื้นที่
    gdf_proj["area_sqkm"] = gdf_proj.geometry.area / 1e6
    print(gdf_proj.nlargest(5, "area_sqkm")[[name_column, "area_sqkm"]])
else:
    print("CRS เป็นแบบฉายภาพแล้ว สามารถคำนวณพื้นที่ได้โดยตรง")
    gdf["area_sqkm"] = gdf.geometry.area / 1e6
    print(gdf.nlargest(5, "area_sqkm")[[name_column, "area_sqkm"]])

EPSG:4326
CRS เป็นแบบภูมิศาสตร์ (Lat/Lon) จะทำการแปลงเป็น Projected CRS (WGS 84 / UTM zone 47N) เพื่อคำนวณพื้นที่ที่แม่นยำขึ้น
        ADM1_TH     area_sqkm
9     เชียงใหม่  22159.519971
28   นครราชสีมา  20750.869676
15    กาญจนบุรี  19436.356316
68          ตาก  17266.546740
71  อุบลราชธานี  15636.863442



## 🔹 ตัวอย่างที่ 5: ใช้ GeoPandas ทำ Spatial Join


In [24]:

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())


   Shape_Leng_left  Shape_Area_left   ADM1_EN_left   ADM1_TH_left  \
0         3.927244         0.275313  Amnat Charoen     อำนาจเจริญ   
1         1.739908         0.079210      Ang Thong        อ่างทอง   
2         2.417227         0.131339        Bangkok  กรุงเทพมหานคร   
3         4.414998         0.340784      Bueng Kan         บึงกาฬ   
4         8.701860         0.844537       Buri Ram      บุรีรัมย์   

  ADM1_PCODE_left ADM1_REF_left ADM1ALT1EN_left ADM1ALT2EN_left  \
0            TH37          None            None            None   
1            TH15          None            None            None   
2            TH10          None            None            None   
3            TH38          None            None            None   
4            TH31          None            None            None   

  ADM1ALT1TH_left ADM1ALT2TH_left  ... ADM1ALT1EN_right ADM1ALT2EN_right  \
0            None            None  ...             None             None   
1            None            N


# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: ใช้ NumPy คำนวณค่า **Mean, Max, Min** ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง
2. **แบบฝึกหัด 2**: ใช้ GeoPandas โหลด **Shapefile** ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด
3. **แบบฝึกหัด 3**: ใช้ GeoPandas ทำ **Spatial Join** ระหว่างข้อมูลจังหวัดและอำเภอ
4. **แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


**แบบฝึกหัด 1: ใช้ NumPy คำนวณค่า Mean, Max, Min ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง**

In [37]:
import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.3745, 0.9507, 0.732], [0.5987, 0.156,  0.156], [0.0581, 0.8662, 0.6011]])
red = np.array([[0.7081, 0.0206, 0.9699], [0.8324, 0.2123, 0.1818], [0.1834, 0.3042, 0.5248]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)

# คำนวณค่าเฉลี่ย
mean_ndvi = np.mean(ndvi)
# คำนวณค่าสูงสุด
max_ndvi = np.max(ndvi)
# คำนวณค่าต่ำสุด
min_ndvi = np.min(ndvi)

print(f"\nค่าเฉลี่ย NDVI: {mean_ndvi:.4f}")
print(f"ค่า NDVI สูงสุด: {max_ndvi:.4f}")
print(f"ค่า NDVI ต่ำสุด: {min_ndvi:.4f}")

ค่า NDVI:
[[-0.30814705  0.95758262 -0.13978495]
 [-0.16330096 -0.15286451 -0.07637655]
 [-0.51884058  0.48017772  0.06776801]]

ค่าเฉลี่ย NDVI: 0.0162
ค่า NDVI สูงสุด: 0.9576
ค่า NDVI ต่ำสุด: -0.5188


**แบบฝึกหัด 2: ใช้ GeoPandas โหลด Shapefile ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด**

In [40]:
import geopandas as gpd

# ข้อมูลจาก OpenDevelopment แม่น้ำโขง
gdf_exercise2 = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

print("CRS ของข้อมูล: ", gdf_exercise2.crs)

# ตรวจสอบว่ามีคอลัมน์ 'ADM1_TH' หรือไม่ หากไม่มี ให้ใช้ 'ADM1_EN'
name_column_exercise2 = 'ADM1_TH' if 'ADM1_TH' in gdf_exercise2.columns else 'ADM1_EN'

if gdf_exercise2.crs.is_geographic:
    print("CRS เป็นแบบภูมิศาสตร์ (Lat/Lon) จะทำการแปลงเป็น Projected CRS (WGS 84 / UTM zone 47N) เพื่อคำนวณพื้นที่ที่แม่นยำขึ้น")
    gdf_proj_exercise2 = gdf_exercise2.to_crs(epsg=32647) # หรือเลือก EPSG code ที่เหมาะสมกับพื้นที่
    gdf_proj_exercise2["area_sqkm"] = gdf_proj_exercise2.geometry.area / 1e6
    print("\nจังหวัดที่มีพื้นที่มากที่สุด 10 อันดับแรก:")
    print(gdf_proj_exercise2.nlargest(10, "area_sqkm")[[name_column_exercise2, "area_sqkm"]])
else:
    print("CRS เป็นแบบฉายภาพแล้ว สามารถคำนวณพื้นที่ได้โดยตรง")
    gdf_exercise2["area_sqkm"] = gdf_exercise2.geometry.area / 1e6
    print("\nจังหวัดที่มีพื้นที่มากที่สุด 10 อันดับแรก:")
    print(gdf_exercise2.nlargest(10, "area_sqkm")[[name_column_exercise2, "area_sqkm"]])

CRS ของข้อมูล:  EPSG:4326
CRS เป็นแบบภูมิศาสตร์ (Lat/Lon) จะทำการแปลงเป็น Projected CRS (WGS 84 / UTM zone 47N) เพื่อคำนวณพื้นที่ที่แม่นยำขึ้น

จังหวัดที่มีพื้นที่มากที่สุด 10 อันดับแรก:
         ADM1_TH     area_sqkm
9      เชียงใหม่  22159.519971
28    นครราชสีมา  20750.869676
15     กาญจนบุรี  19436.356316
68           ตาก  17266.546740
71   อุบลราชธานี  15636.863442
66  สุราษฎร์ธานี  13075.460617
22    แม่ฮ่องสอน  12765.285306
7        ชัยภูมิ  12634.334796
18         ลำปาง  12487.598360
41     เพชรบูรณ์  12395.386041


**แบบฝึกหัด 3: ใช้ GeoPandas ทำ Spatial Join ระหว่างข้อมูลจังหวัดและอำเภอ**

In [47]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลอำเภอ (ADM2) ของประเทศไทย
# ใช้ URL จาก GADM ที่รวมทุกระดับชั้นข้อมูล แล้ว GeoPandas จะเลือกชั้นข้อมูลที่เหมาะสมให้
gdf_districts_exercise3 = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_THA_shp.zip", layer="gadm41_THA_2")

print("ข้อมูลอำเภอ 5 แถวแรก:")
print(gdf_districts_exercise3.head())

# ตรวจสอบ CRS ของข้อมูลอำเภอ
print(f"\nCRS ของข้อมูลอำเภอ: {gdf_districts_exercise3.crs}")

ข้อมูลอำเภอ 5 แถวแรก:
       GID_2 GID_0   COUNTRY    GID_1         NAME_1          NL_NAME_1  \
0  THA.1.1_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
1  THA.1.2_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
2  THA.1.3_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
3  THA.1.4_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
4  THA.1.5_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   

                NAME_2 VARNAME_2             NL_NAME_2  TYPE_2 ENGTYPE_2  \
0             Chanuman        NA          อำเภอชานุมาน  Amphoe  District   
1           Hua Taphan        NA         อำเภอหัวตะพาน  Amphoe  District   
2             Lu Amnat        NA         อำเภอลืออำนาจ  Amphoe  District   
3  Muang Amnat Charoen        NA  อำเภอเมืองอำนาจเจริญ  Amphoe  District   
4     Pathum Ratwongsa        NA      อำเภอปทุมราชวงศา  Amphoe  District   

   CC_2    HASC_2                                           geometry  

In [48]:
# ทำ Spatial Join ระหว่างข้อมูลอำเภอกับข้อมูลจังหวัด
# เราจะใช้ gdf_districts_exercise3 เป็น 'left' layer (อำเภอ) และ gdf_exercise2 เป็น 'right' layer (จังหวัด)
# 'predicate="within"' ใช้สำหรับหาว่าอำเภอใดบ้างที่ 'อยู่ภายใน' ขอบเขตของจังหวัด
gdf_joined_exercise3 = gpd.sjoin(gdf_districts_exercise3, gdf_exercise2, how="inner", predicate="within")

print("ตัวอย่างข้อมูลที่เชื่อมโยงกัน (5 แถวแรก) แสดงอำเภอพร้อมข้อมูลจังหวัดที่อำเภอนั้นสังกัด:")
display(gdf_joined_exercise3.head())

ตัวอย่างข้อมูลที่เชื่อมโยงกัน (5 แถวแรก) แสดงอำเภอพร้อมข้อมูลจังหวัดที่อำเภอนั้นสังกัด:


,GID_2,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,NAME_2,VARNAME_2,NL_NAME_2,TYPE_2,...,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo
15,THA.3.2_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Kapi,NA,บางกะป,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
16,THA.3.3_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Khae,NA,บางแค,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
17,THA.3.4_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Khen,NA,บางเขน,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
18,THA.3.5_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Kho Laem,NA,บางคอแหลม,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
21,THA.3.8_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Rak,NA,บางรัก,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT


**แบบฝึกหัด 4: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด**

In [55]:
import numpy as np
import geopandas as gpd

# ตรวจสอบว่า gdf_exercise2 มีอยู่ในหน่วยความจำหรือไม่ ถ้าไม่มี ให้โหลดใหม่
# โดยปกติแล้วหลังจากรันแบบฝึกหัด 2, gdf_exercise2 ควรจะถูกโหลดไว้แล้ว
if 'gdf_exercise2' not in locals():
    print("กำลังโหลดข้อมูลจังหวัดใหม่...")
    gdf_exercise2 = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

# สร้างค่า NDVI เฉลี่ยจำลองสำหรับแต่ละจังหวัด
# ค่า NDVI โดยทั่วไปอยู่ระหว่าง -1 ถึง 1
# สร้างจำนวนค่าเท่ากับจำนวนจังหวัด
num_provinces = len(gdf_exercise2)
avg_ndvi_values = np.random.uniform(low=-1.0, high=1.0, size=num_provinces)

# เพิ่มค่า NDVI เฉลี่ยจำลองเข้าไปใน GeoDataFrame
gdf_exercise2['avg_ndvi'] = avg_ndvi_values

# ตรวจสอบว่ามีคอลัมน์ 'ADM1_TH' หรือไม่ หากไม่มี ให้ใช้ 'ADM1_EN'
name_column_exercise4 = 'ADM1_TH' if 'ADM1_TH' in gdf_exercise2.columns else 'ADM1_EN'

# ค้นหาจังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด
province_highest_ndvi = gdf_exercise2.nlargest(1, 'avg_ndvi')

print("\nจังหวัดที่มีค่า NDVI เฉลี่ยสูงสุดคือ:")
print(f"  {province_highest_ndvi[name_column_exercise4].iloc[0]} (NDVI: {province_highest_ndvi['avg_ndvi'].iloc[0]:.4f})")

print("\n5 จังหวัดแรกที่มีค่า NDVI เฉลี่ยสูงสุด:")
display(gdf_exercise2.nlargest(5, 'avg_ndvi')[[name_column_exercise4, 'avg_ndvi']])


จังหวัดที่มีค่า NDVI เฉลี่ยสูงสุดคือ:
  ประจวบคีรีขันธ์ (NDVI: 0.9994)

5 จังหวัดแรกที่มีค่า NDVI เฉลี่ยสูงสุด:


,ADM1_TH,avg_ndvi
49,ประจวบคีรีขันธ์,0.999435
50,ระนอง,0.993274
36,ปทุมธานี,0.974552
17,กระบี่,0.972421
16,ขอนแก่น,0.948790
